# 09 · Hosted agents – when configuration is not enough

A **prompt agent** is configuration. A **hosted agent** is *your code* – any framework, any library, custom logic – packaged and run by Foundry with an endpoint, an Entra identity, scaling, tracing and session storage.

| | Prompt agent | Hosted agent |
|---|---|---|
| You write | definition (model, instructions, tools) | code (Agent Framework, LangGraph, your own) |
| Runs | inside Foundry's agent runtime | in a managed container/sandbox |
| Tools | built-in, MCP, A2A, functions you loop yourself | anything your code can call |
| Deploy | `create_version` | `azd deploy` (code ZIP or container) |
| Protocols | Responses | Responses, Invocations, A2A, ... |

⚠️ Deploying needs the **Foundry Project Manager** role on the project (formerly *Azure AI Project Manager*). `python scripts/grant_hosted_role.py` assigns it to you. Hosted agents are a newer feature: check the [docs](https://learn.microsoft.com/azure/foundry/agents/quickstarts/quickstart-hosted-agent) if a command differs. Deep dive: [docs/hosted-agents.md](../docs/hosted-agents.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

print((ROOT / "src" / "jarvis-hosted" / "main.py").read_text())

## What the code does
1. `FoundryChatClient` talks to the model through the project (identity: yours locally, the **agent identity** when hosted).
2. `Agent` adds instructions, the local tool `workshop_status` and the skills from `skills/` (direct injection, chapter 07): every call carries every skill. Section 4 deploys a second agent that loads skills on demand instead.
3. `ResponsesHostServer` exposes the agent over the **Responses protocol**: Foundry stores conversations and handles streaming.

## 1 · Install the azd extension and run locally
```bash
python scripts/grant_hosted_role.py          # once: Foundry Project Manager role for you
azd extension install microsoft.foundry      # adds: azd ai agent ...
python scripts/sync_skills.py                # copy skills/ into src/jarvis-hosted/skills
azd ai agent run                             # starts main.py locally and opens the agent inspector
azd ai agent invoke --local "Status of the arc reactor?"
```

## 2 · Deploy
`azd deploy` zips `src/jarvis-hosted`, Foundry builds it remotely (`codeConfiguration` in [azure.yaml](../azure.yaml)) and creates version 1. No Docker needed in code mode.

In [ ]:
sh("azd deploy jarvis-hosted --no-prompt")

## 3 · Call it like any agent
Hosted agents answer the same Responses API as prompt agents.

In [ ]:
hosted = "jarvis-hosted"
oai = openai_client(agent_name=hosted)
conv = oai.conversations.create()
for q in ["What is the status of the arc reactor?", "A suit just started smoking in the hangar. What do I do?"]:
    r = oai.responses.create(conversation=conv.id, input=q)
    print(f"> {q}\n{r.output_text}\n")

The second question should follow the **stark-escalation** skill ("Severity: critical", evacuate, notify Pepper) – proof the skill reached the container.

## 4 · Skills on demand: `jarvis-skills`
[src/jarvis-skills/main.py](../src/jarvis-skills/main.py) is the same J.A.R.V.I.S. with one change: skills go through Agent Framework's **`SkillsProvider`** instead of into the instructions.

| | `jarvis-hosted` | `jarvis-skills` |
|---|---|---|
| Skills reach the model | all of them, pasted into the instructions | name and description only; the body when the model calls `load_skill` |
| Workshop status | the tool `workshop_status` | the code-defined skill `workshop-status` (a resource and a script) |
| Cost per call | grows with every skill you add | grows only with the skills the model loads |
| Proof the skill was used | the answer | the answer, plus `load_skill` calls in the response output and the traces |

All three skill tools (`load_skill`, `read_skill_resource`, `run_skill_script`) need approval by default. The agent turns approval off because the skills are this repo's own. Keep it on for skills from elsewhere.

> 💡 To draft a skill without deploying, run `python samples/skills_provider.py "your question"`: the same agent as a plain script, printing which skill tools the model called.


In [ ]:
sh("azd deploy jarvis-skills --no-prompt")

In [ ]:
skills_oai = openai_client(agent_name="jarvis-skills")
for q in ["What is the status of the arc reactor?", "A suit just started smoking in the hangar. What do I do?"]:
    r = skills_oai.responses.create(input=q)
    calls = [f"{o.name} {o.arguments}" for o in r.output if o.type == "function_call"]
    print(f"> {q}")
    print("  skill tools:", calls or "none")
    print(f"{r.output_text}\n")

Expect `load_skill` with `workshop-status` for the first question and `stark-escalation` for the second. The second answer should carry the same `Severity: critical` marker as `jarvis-hosted`.

> ❓ **Not run against a live project yet.** The code was tested locally: `ResponsesHostServer` returned the `function_call` items for `load_skill`, `read_skill_resource` and `run_skill_script` with a scripted model. Whether a real model picks the right skill, and whether the Foundry endpoint passes those items through unchanged, still needs a deployed run.


## 5 · Observe it
```bash
azd ai agent monitor --follow     # container logs
azd ai agent show                 # versions, endpoint, protocols
```
Hosted agents get `APPLICATIONINSIGHTS_CONNECTION_STRING` from the platform, so their traces land in the same Application Insights as chapter 03 – find the conversation id in **Traces**.

## 👀 In the portal
🧪 **Build → Agents → jarvis-hosted**: a *hosted* agent. Its page lists the versions that `azd deploy` created, the playground and **Traces** with the conversation id.

**Build → Agents → jarvis-skills**: the same, and its **Traces** should show the `load_skill` tool call when the model loads a skill (❓ not checked on a live project).

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
oai.conversations.delete(conv.id)
# delete_agent("jarvis-hosted")   # uncomment to remove the hosted agent; `azd down --purge` removes everything
# delete_agent("jarvis-skills")   # uncomment to remove the skills-on-demand agent

## 🎯 Recap
Prompt agents for speed and simplicity; hosted agents when you need code. Skills ride along either way: pasted into the instructions (`jarvis-hosted`) or loaded on demand by `SkillsProvider` (`jarvis-skills`). Both are first-class Foundry agents: same conversations, tracing, and evaluations (next act).

> 💡 Next: Act III – **F.R.I.D.A.Y.** proves it works. [10 · Datasets and evaluations](10_datasets_and_evals.ipynb).